## Chat Completions with Frontier Models


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

response = client.responses.create(
    model="gpt-5-mini",
    input="Write a one-sentence bedtime story about a unicorn.",
)

print(response.output_text)

Under the silver moon, a sleepy unicorn with starlight in her mane tiptoed through a meadow of whispering flowers, carrying a child's wish on her horn until dawn painted the world with gentle gold.


In [ ]:
from anthropic import Anthropic

client = Anthropic()

message = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": "What should I search for to find the latest developments in renewable energy?",
        }
    ],
)

for block in message.content:
    if block.type == "text":
        print(block.text)

# Search suggestions for renewable energy developments:

## Specific technology searches:
- "Solar panel efficiency 2024"
- "Offshore wind energy developments"
- "Green hydrogen production advances"
- "Battery storage technology breakthrough"
- "Perovskite solar cells latest"
- "Floating wind farms"

## Broad industry searches:
- "Renewable energy news 2024"
- "Clean energy investments"
- "Grid-scale energy storage"
- "Renewable energy capacity additions"

## Policy and market searches:
- "Renewable energy policy updates"
- "Clean energy subsidies [your country]"
- "Corporate renewable energy deals"
- "Net zero commitments"

## Reliable sources to check:
- International Energy Agency (IEA) reports
- BloombergNEF
- MIT Technology Review (Energy section)
- Renewable Energy World
- GTM/Green Tech Media
- Nature Energy journal

**Tip:** Add the current year or "latest" to searches, and use Google News or specialized databases for the most recent updates.

What specific area of renewable en

## Interacting with Ollama (open source llms)

In [ ]:
from openai import OpenAI

ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = ollama_client.chat.completions.create(
    model="gemma3:1b",
    messages=[{"role": "user", "content": "Write a short poem about the color blue"}]
)

print(response.choices[0].message.content)

Okay carefully, breathe, and feel this hue,
A gentle ripple, fresh and cool and true.
Like summer sky, a dreamy, vibrant grace, 
Lost in blues, in a tranquil space.

It speaks of ocean depths and distant sight,
A velvet blanket in the fading light. 



Would you like me to try one with a different poetic style or length?


In [ ]:
import os
print(os.getcwd())

c:\Users\sivag\git-projects\gradio-multimodal-chat-exploration


In [ ]:
models = groq_client.models.list()
for m in models.data:
    print(m.id)

meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
whisper-large-v3-turbo
whisper-large-v3
groq/compound
groq/compound-mini
canopylabs/orpheus-v1-english
allam-2-7b
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b


In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()


groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Write a short poem about the color blue"}]
)

print(response.choices[0].message.content)

In a hush of quiet skies,  
Blue drips like a gentle sigh—  
A ripple from the sea, a dream,  
A promise held in endless stream.  

It paints the dawn, it cradles night,  
A velvet hush of soft delight—  
Blue, the quiet in our sight,  
A world that glows in quiet light.


## OpenAI-Claude debate loop


In [ ]:
from openai import OpenAI
from anthropic import Anthropic

openai_client = OpenAI()
anthropic_client = Anthropic()

topic = "Is remote work better than office work?"

openai_history = [
    {"role": "user", "content": f"Let's debate: {topic}. Give your opening argument in 2-3 sentences."}
]
claude_history = []

for turn in range(4):
    response = openai_client.chat.completions.create(model="gpt-5-mini", messages=openai_history)
    openai_reply = response.choices[0].message.content
    print(f"OpenAI: {openai_reply}\n")

    openai_history.append({"role": "assistant", "content": openai_reply})
    claude_history.append({"role": "user", "content": openai_reply})

    claude_response = anthropic_client.messages.create(
        model="claude-sonnet-4-5", max_tokens=200, messages=claude_history
    )
    claude_reply = claude_response.content[0].text
    print(f"Claude: {claude_reply}\n")

    claude_history.append({"role": "assistant", "content": claude_reply})
    openai_history.append({"role": "user", "content": claude_reply})

=================

In [ ]:
#tool calling into the Gradio chat UI
def chat(message, history):
    messages = [{"role": "system", "content": "You are a helpful airline assistant."}]
    messages += history
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(model="gpt-5-mini", messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        messages.append(response.choices[0].message)

        for tool_call in response.choices[0].message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            city = arguments["destination_city"]
            price = get_ticket_price(city)

            messages.append({
                "role": "tool",
                "content": json.dumps({"destination_city": city, "price": price}),
                "tool_call_id": tool_call.id
            })

        response = client.chat.completions.create(model="gpt-5-mini", messages=messages)

    return response.choices[0].message.content

demo = gr.ChatInterface(fn=chat)
demo.launch()

In [ ]:
#Tool calling with SQLite
import sqlite3

conn = sqlite3.connect("tickets.db")
cursor = conn.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price TEXT)")
cursor.executemany(
    "INSERT OR REPLACE INTO prices VALUES (?, ?)",
    [("london", "£299"), ("paris", "£210"), ("tokyo", "£850")]
)
conn.commit()

def get_ticket_price(destination_city):
    cursor.execute("SELECT price FROM prices WHERE city = ?", (destination_city.lower(),))
    row = cursor.fetchone()
    return row[0] if row else "Unknown destination"

In [ ]:
#Agentic AI: multi-tool workflows
def check_seat_availability(destination_city):
    availability = {"london": "12 seats", "paris": "3 seats", "tokyo": "sold out"}
    return availability.get(destination_city.lower(), "Unknown")

seat_function = {
    "name": "check_seat_availability",
    "description": "Check how many seats are available on flights to a destination city.",
    "parameters": {
        "type": "object",
        "properties": {"destination_city": {"type": "string", "description": "The destination city"}},
        "required": ["destination_city"]
    }
}

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": seat_function}
]

In [ ]:
#Agentic AI: multi-tool workflows
def check_seat_availability(destination_city):
    availability = {"london": "12 seats", "paris": "3 seats", "tokyo": "sold out"}
    return availability.get(destination_city.lower(), "Unknown")

seat_function = {
    "name": "check_seat_availability",
    "description": "Check how many seats are available on flights to a destination city.",
    "parameters": {
        "type": "object",
        "properties": {"destination_city": {"type": "string", "description": "The destination city"}},
        "required": ["destination_city"]
    }
}

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": seat_function}
]

In [ ]:
#Multimodal: DALL-E 3 + Text-to-Speech + Blocks
def generate_image(prompt):
    result = client.images.generate(model="dall-e-3", prompt=prompt, size="1024x1024", n=1)
    return result.data[0].url

def generate_speech(text):
    response = client.audio.speech.create(model="tts-1", voice="alloy", input=text)
    response.stream_to_file("output.mp3")
    return "output.mp3"

In [7]:
from openai import OpenAI
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

openai_client = OpenAI()
anthropic_client = Anthropic()

topic = "Is remote work better than office work"

openai_history = [{
    "role": "user",
    "content": f"Let's debate: {topic}. Give your opening argument in 2-3 sentences."
}]
claude_history = []

for turn in range(4):
    response = openai_client.chat.completions.create(model="gpt-5-mini", messages=openai_history)
    openai_reply = response.choices[0].message.content
    print(f"OpenAI: {openai_reply}\n")

    openai_history.append({"role": "assistant", "content": openai_reply})
    claude_history.append({"role": "user", "content": openai_reply})

    claude_response = anthropic_client.messages.create(
        model="claude-sonnet-4-5", max_tokens=200, messages=claude_history
    )
    claude_reply = claude_response.content[0].text
    print(f"Claude: {claude_reply}\n")

    claude_history.append({"role": "assistant", "content": claude_reply})
    openai_history.append({"role": "user", "content": claude_reply})

OpenAI: Remote work is better because it lets people work when and where they’re most productive, eliminates commutes that drain time and energy, and improves work–life balance. It also reduces employer overhead and expands the talent pool beyond geographic limits, enabling companies to hire and retain better-fit employees. For many roles and organizations, those efficiency, cost, and satisfaction gains outweigh the benefits of being physically co-located.

Claude: This is a solid case for remote work, but let me push back on a few points:

**Where the argument is strongest:**
- Commute elimination is an unambiguous win for most people
- Geographic flexibility genuinely expands talent pools
- Overhead savings are real for companies willing to fully commit

**Where it's more complicated:**

*"Work when and where they're most productive"* – This assumes people can accurately judge their own productivity and that asynchronous work doesn't create costly delays. Many people *feel* productiv

In [9]:
import gradio as gr 

def greet(name):
    return f"Hello, {name}!"

demo = gr.Interface(fn = greet, inputs="text",outputs="text")
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [16]:
#Interface with an llm callback
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI()

def chat_with_gpt(message):
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages = [{"role":"user","content":message}]
    )
    return response.choices[0].message.content

demo = gr.Interface(fn=chat_with_gpt,inputs="text",outputs="text")
demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [17]:
from click import prompt
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI()

def summarize_pros_cons(topic,num_points):
    prompt = f"List{num_points} pros and cons of :{topic}"
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages = [{"role":"user","content":prompt}]
    )
    return response.choices[0].message.content

demo = gr.Interface(fn=summarize_pros_cons,inputs=["text",gr.Slider(1,5,value=3,step=1)],outputs="text")
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [1]:
#Streaming Responses (Generator Functions)

import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI()
def chat_stream(message):
    stream = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": message}],
        stream=True
    )
    partial_reply = ""
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            partial_reply += content
            yield partial_reply
demo = gr.Interface(fn=chat_stream, inputs="text", outputs="text")
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Chat UI, Memory and Prompting

In [2]:
#ChatInterface with Real Conversation Memory

import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI()
def chat(message, history):
    # history is everything BEFORE this turn, already shaped as
    # [{"role": "user"/"assistant", "content": "..."}]
    messages = history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages
    )
    return response.choices[0].message.content
demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
##ChatInterface with Real Conversation Memory with system Prompts
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI()
def chat(message, history):
    system_message = {
        "role": "system",
        "content": "You are a sarcastic assistant who answers correctly but with dry wit."
    }
    messages = [system_message] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages
    )
    return response.choices[0].message.content
demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [4]:
#Few-Shot (Multi-Shot) Prompting

import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI()
def chat(message, history):
    system_message = {
        "role": "system",
        "content": "You convert casual sentences into formal business English."
    }
    examples = [
        {"role": "user", "content": "hey can u send me that file"},
        {"role": "assistant", "content": "Could you please send me that file at your earliest convenience?"},
        {"role": "user", "content": "gonna be late sry"},
        {"role": "assistant", "content": "I apologize, but I will be arriving later than scheduled."}
    ]
    messages = [system_message] + examples + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
    return response.choices[0].message.content
demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Tool / Function Calling

In [5]:
# Tool / Function Calling


def get_ticket_price(destination_city):
    prices = {"london": "£299", "paris": "£210", "tokyo": "£850"}
    city = destination_city.lower()
    return prices.get(city, "Unknown destination")
# Confirm it works standalone, before any API/tool-calling syntax is involved
print(get_ticket_price("london"))

£299


In [6]:
## Describing the Function to the Model
price_function = {
    "name": "get_ticket_price",
    "description": (
        "Get the price of a return ticket to the destination city. "
        "Call this whenever a user asks about ticket prices, "
        "for example 'How much is a ticket to Paris?'"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to"
            }
        },
        "required": ["destination_city"]
    }
}
tools = [{"type": "function", "function": price_function}]


In [7]:
 ##The Full Call fi Tool-Call fi Result fi Final-Answer Loop
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI()
def get_ticket_price(destination_city):
    prices = {"london": "£299", "paris": "£210", "tokyo": "£850"}
    city = destination_city.lower()
    return prices.get(city, "Unknown destination")
price_function = {
    "name": "get_ticket_price",
    "description": (
        "Get the price of a return ticket to the destination city. "
        "Call this whenever a user asks about ticket prices."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to"
            }
        },
        "required": ["destination_city"]
    }
}
tools = [{"type": "function", "function": price_function}]
messages = [{"role": "user", "content": "How much is a ticket to Tokyo?"}]
# Step 1: ask the model, offering it the tool
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
    tools=tools
)
print(response.choices[0].finish_reason)          # "tool_calls" when a tool is requested
print(response.choices[0].message.tool_calls)     # list of requested calls
# Step 2: actually execute the requested function
tool_call = response.choices[0].message.tool_calls[0]
arguments = json.loads(tool_call.function.arguments)  # arguments arrive as a JSON string
city = arguments["destination_city"]
price = get_ticket_price(city)
# Step 3: hand the model's own tool-call message, and the real result, back to it
messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "content": json.dumps({"destination_city": city, "price": price}),
    "tool_call_id": tool_call.id
})
# Step 4: ask again -- now the model can write a natural-language final answer
final_response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
print(final_response.choices[0].message.content)



tool_calls
[ChatCompletionMessageFunctionToolCall(id='call_ZtNqxRwpxTPilP8z4AxW33lt', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function')]
A ticket to Tokyo costs £850. 

Would you like that price for a one-way or round-trip ticket, travel dates, departure airport, or seat class so I can find the best options or book it for you?


In [8]:
#Tool Calling Inside a Gradio Chat UI

import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI()
def get_ticket_price(destination_city):
    prices = {"london": "£299", "paris": "£210", "tokyo": "£850"}
    city = destination_city.lower()
    return prices.get(city, "Unknown destination")
price_function = {
    "name": "get_ticket_price",
    "description": (
        "Get the price of a return ticket to the destination city. "
        "Call this whenever a user asks about ticket prices."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to"
            }
        },
        "required": ["destination_city"]
    }
}
tools = [{"type": "function", "function": price_function}]
def chat(message, history):
    messages = [{"role": "system", "content": "You are a helpful airline assistant."}]
    messages += history
    messages.append({"role": "user", "content": message})
    response = client.chat.completions.create(model="gpt-5-mini", messages=messages, tools=tools)
    if response.choices[0].finish_reason == "tool_calls":
        messages.append(response.choices[0].message)
        for tool_call in response.choices[0].message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            city = arguments["destination_city"]
            price = get_ticket_price(city)
            messages.append({
                "role": "tool",
                "content": json.dumps({"destination_city": city, "price": price}),
                "tool_call_id": tool_call.id
            })
        response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
    return response.choices[0].message.content
demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [9]:
# Backing a Tool with a Real SQLite Database

import sqlite3
conn = sqlite3.connect("tickets.db")
cursor = conn.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price TEXT)")
cursor.executemany(
    "INSERT OR REPLACE INTO prices VALUES (?, ?)",
    [("london", "£299"), ("paris", "£210"), ("tokyo", "£850")]
)
conn.commit()
def get_ticket_price(destination_city):
    cursor.execute("SELECT price FROM prices WHERE city = ?", (destination_city.lower(),))
    row = cursor.fetchone()
    return row[0] if row else "Unknown destination"
# Quick standalone check
print(get_ticket_price("Paris"))

£210


In [10]:
#Agentic Workflows and Multimodal

import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI()
def get_ticket_price(destination_city):
    prices = {"london": "£299", "paris": "£210", "tokyo": "£850"}
    return prices.get(destination_city.lower(), "Unknown destination")
def check_seat_availability(destination_city):
    availability = {"london": "12 seats", "paris": "3 seats", "tokyo": "sold out"}
    return availability.get(destination_city.lower(), "Unknown")
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {"destination_city": {"type": "string", "description": "The destination city"}},
        "required": ["destination_city"]
    }
}
seat_function = {
    "name": "check_seat_availability",
    "description": "Check how many seats are available on flights to a destination city.",
    "parameters": {
        "type": "object",
        "properties": {"destination_city": {"type": "string", "description": "The destination city"}},
        "required": ["destination_city"]
    }
}
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": seat_function}
]
def chat(message, history):
    messages = [{"role": "system", "content": "You are a helpful airline assistant."}]
    messages += history
    messages.append({"role": "user", "content": message})
    response = client.chat.completions.create(model="gpt-5-mini", messages=messages, tools=tools)
    if response.choices[0].finish_reason == "tool_calls":
        messages.append(response.choices[0].message)
        for tool_call in response.choices[0].message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            city = arguments["destination_city"]
            if tool_call.function.name == "get_ticket_price":
                result = get_ticket_price(city)
            elif tool_call.function.name == "check_seat_availability":
                result = check_seat_availability(city)
            else:
                result = "Unknown tool"
            messages.append({
                "role": "tool",
                "content": json.dumps({"destination_city": city, "result": result}),
                "tool_call_id": tool_call.id
            })
        response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
    return response.choices[0].message.content
demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Image Generation


In [13]:
import base64

def generate_image(prompt):
    result = client.images.generate(
        model="gpt-image-1",
        prompt=prompt,
        size="1024x1024",
        n=1
    )
    image_base64 = result.data[0].b64_json
    image_bytes = base64.b64decode(image_base64)

    with open("generated_image.png", "wb") as f:
        f.write(image_bytes)

    return "generated_image.png"

image_path = generate_image("A cozy reading nook by a rainy window, watercolor style")
print(image_path)

generated_image.png


In [12]:
models = client.models.list()
for m in models.data:
    if "image" in m.id or "dall" in m.id.lower() or "gpt-image" in m.id:
        print(m.id)

gpt-image-1
gpt-image-1-mini
gpt-image-1.5
chatgpt-image-latest
gpt-image-2
gpt-image-2-2026-04-21
gpt-image-2.5-flare
gpt-image-2.5-sunburst
gpt-image-2.5-flare-2026-09-08
gpt-image-2.5-sunburst-2026-09-08


##Text-to-Speech (TTS

In [14]:
#Text-to-Speech (TTS)
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI()
def generate_speech(text, out_path="output.mp3"):
    response = client.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input=text
    )
    response.stream_to_file(out_path)
    return out_path
audio_file = generate_speech("Welcome aboard. Your ticket has been confirmed.")
print(audio_file)

output.mp3


C:\Users\sivag\AppData\Local\Temp\ipykernel_27624\2568425212.py:12: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(out_path)


## Combined Multimodal Gradio App

In [19]:
import json
import base64
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()

def get_ticket_price(destination_city):
    prices = {"london": "£299", "paris": "£210", "tokyo": "£850"}
    return prices.get(destination_city.lower(), "Unknown destination")

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {"destination_city": {"type": "string", "description": "The destination city"}},
        "required": ["destination_city"]
    }
}
tools = [{"type": "function", "function": price_function}]

def chat(message, history):
    messages = [{"role": "system", "content": "You are a helpful airline assistant."}]
    messages += history
    messages.append({"role": "user", "content": message})
    response = client.chat.completions.create(model="gpt-5-mini", messages=messages, tools=tools)
    if response.choices[0].finish_reason == "tool_calls":
        messages.append(response.choices[0].message)
        for tool_call in response.choices[0].message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            city = arguments["destination_city"]
            price = get_ticket_price(city)
            messages.append({
                "role": "tool",
                "content": json.dumps({"destination_city": city, "price": price}),
                "tool_call_id": tool_call.id
            })
        response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
    return response.choices[0].message.content

def generate_image(prompt):
    result = client.images.generate(model="gpt-image-1", prompt=prompt, size="1024x1024", n=1)
    image_bytes = base64.b64decode(result.data[0].b64_json)
    with open("generated_image.png", "wb") as f:
        f.write(image_bytes)
    return "generated_image.png"

def generate_speech(text, out_path="output.mp3"):
    response = client.audio.speech.create(model="tts-1", voice="alloy", input=text)
    response.stream_to_file(out_path)
    return out_path

with gr.Blocks() as demo:
    with gr.Row():
        chatbot = gr.Chatbot()
        image_output = gr.Image()
    msg = gr.Textbox(label="Message")
    audio_output = gr.Audio()

    def respond(message, history):
        reply = chat(message, history)
        new_history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply}
        ]
        image_path = generate_image(message)
        audio_path = generate_speech(reply)
        return new_history, "", image_path, audio_path

    msg.submit(respond, [msg, chatbot], [chatbot, msg, image_output, audio_output])

demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
